In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import numpy as np
import copy

carico il subset dei commenti che ho taggato violenti e li analizzo.

prima divido per tossicità (3 livelli), poi applico lo split dei dati e infine i modelli di regressione

In [2]:
df = pd.read_csv(
    "../datasets/dataset_4_violence.csv",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
    engine="python"
)
df

,commento,valutazione,tossicita,tipo_tossicita,age_group,gender,utente_id,comment_cleaned,emoji_set,emoji_count,...,comment_tokenized,comment_tok=2,comment_tok=3,comment_lemmatized,VAD_valence,VAD_arousal,VAD_dominance,bad_words_flag,bad_words_count,bad_words_matches
0,Think carefully. Accuse me of raping kids and ...,Extremely violent,3,4,1,1,1,think carefully accuse raping kids going rape ...,[],0,...,"['think', 'carefully', 'accuse', 'raping', 'ki...","[('think', 'carefully'), ('carefully', 'accuse...","[('think', 'carefully', 'accuse'), ('carefully...","['think', 'carefully', 'accuse', 'rap', 'kid',...",0.074000,0.158000,0.014857,1,2,"raping, rape"
1,Maybe someone should choke you and your immedi...,Extremely violent,3,4,1,1,1,maybe someone choke immediate family death see...,[],0,...,"['maybe', 'someone', 'choke', 'immediate', 'fa...","[('maybe', 'someone'), ('someone', 'choke'), (...","[('maybe', 'someone', 'choke'), ('someone', 'c...","['maybe', 'someone', 'choke', 'immediate', 'fa...",0.062500,-0.008375,0.006875,0,0,NaN
2,I will murder your family friend!,Extremely violent,3,4,1,1,1,murder family friend,[],0,...,"['murder', 'family', 'friend']","[('murder', 'family'), ('family', 'friend')]","[('murder', 'family', 'friend')]","['murder', 'family', 'friend']",0.260667,0.164667,0.215333,0,0,NaN
3,"Eww, refer to me as a disgusting subhuman Celt...",Extremely violent,3,4,1,1,1,eww refer disgusting subhuman celt blood eagle...,[],0,...,"['eww', 'refer', 'disgusting', 'subhuman', 'ce...","[('eww', 'refer'), ('refer', 'disgusting'), ('...","[('eww', 'refer', 'disgusting'), ('refer', 'di...","['eww', 'refer', 'disgust', 'subhuman', 'celt'...",-0.236333,0.039333,0.097333,0,0,NaN
4,And then he rapes you,Extremely violent,3,4,1,1,1,rapes,[],0,...,['rapes'],[],[],['rape'],0.000000,0.000000,0.000000,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1062,Commies are going to commie. These people hate...,Extremely violent,3,4,1,0,211,commies going commie people hate want die,[],0,...,"['commies', 'going', 'commie', 'people', 'hate...","[('commies', 'going'), ('going', 'commie'), ('...","[('commies', 'going', 'commie'), ('going', 'co...","['commie', 'go', 'commie', 'people', 'hate', '...",-0.329714,0.375857,0.045000,0,0,NaN
1063,Yeah but then the good guys win and rape you t...,Extremely violent,3,4,1,0,211,yeah good guys win rape death occupy nation in...,[],0,...,"['yeah', 'good', 'guys', 'win', 'rape', 'death...","[('yeah', 'good'), ('good', 'guys'), ('guys', ...","[('yeah', 'good', 'guys'), ('good', 'guys', 'w...","['yeah', 'good', 'guy', 'win', 'rape', 'death'...",0.012750,0.015250,0.121125,1,1,rape
1064,If you're white and use your gun to defend you...,A little violent,1,4,1,0,211,white use gun defend arrested racist get life ...,[],0,...,"['white', 'use', 'gun', 'defend', 'arrested', ...","[('white', 'use'), ('use', 'gun'), ('gun', 'de...","[('white', 'use', 'gun'), ('use', 'gun', 'defe...","['white', 'use', 'gun', 'defend', 'arrest', 'r...",-0.115429,0.365714,0.212857,0,0,NaN
1065,I will rape Elon Musk,A little violent,1,4,1,0,211,rape elon musk,[],0,...,"['rape', 'elon', 'musk']","[('rape', 'elon'), ('elon', 'musk')]","[('rape', 'elon', 'musk')]","['rape', 'elon', 'musk']",-0.469000,0.310000,-0.020000,1,1,rape


In [ ]:
# Definisco le 3 soglie di sensibilità

# Poco sensibile: solo estremamente tossico è 1
df["tox_poco_sensibile"] = df["tossicita"].apply(lambda x: 1 if x == 3 else 0)

# Media sensibilità: tossico o estremamente tossico
df["tox_medio_sensibile"] = df["tossicita"].apply(lambda x: 1 if x >= 2 else 0)

# Molto sensibile: tutto tranne non tossico
df["tox_molto_sensibile"] = df["tossicita"].apply(lambda x: 1 if x >= 1 else 0)


In [11]:
# Funzione per generare una colonna isToxic in base al livello di sensibilità
def imposta_isToxic(df, sensibilita):
    if sensibilita == "poco":
        df["isToxic"] = df["tossicita"].apply(lambda x: 1 if x == 3 else 0)
    elif sensibilita == "media":
        df["isToxic"] = df["tossicita"].apply(lambda x: 1 if x >= 2 else 0)
    elif sensibilita == "alta":
        df["isToxic"] = df["tossicita"].apply(lambda x: 1 if x >= 1 else 0)
    return df

# Crea 3 versioni del dataset
df_poca_sens = imposta_isToxic(df.copy(), sensibilita="poco")
df_media_sens = imposta_isToxic(df.copy(), sensibilita="media")
df_alta_sens = imposta_isToxic(df.copy(), sensibilita="alta")

In [16]:
df_poca_sens[["tossicita", "isToxic"]].groupby("isToxic").count()

,tossicita
isToxic,
0,420
1,647


In [12]:
df_poca_sens.head()

,commento,valutazione,tossicita,tipo_tossicita,age_group,gender,utente_id,comment_cleaned,emoji_set,emoji_count,...,comment_tok=2,comment_tok=3,comment_lemmatized,VAD_valence,VAD_arousal,VAD_dominance,bad_words_flag,bad_words_count,bad_words_matches,isToxic
0,Think carefully. Accuse me of raping kids and ...,Extremely violent,3,4,1,1,1,think carefully accuse raping kids going rape ...,[],0,...,"[('think', 'carefully'), ('carefully', 'accuse...","[('think', 'carefully', 'accuse'), ('carefully...","['think', 'carefully', 'accuse', 'rap', 'kid',...",0.074000,0.158000,0.014857,1,2,"raping, rape",1
1,Maybe someone should choke you and your immedi...,Extremely violent,3,4,1,1,1,maybe someone choke immediate family death see...,[],0,...,"[('maybe', 'someone'), ('someone', 'choke'), (...","[('maybe', 'someone', 'choke'), ('someone', 'c...","['maybe', 'someone', 'choke', 'immediate', 'fa...",0.062500,-0.008375,0.006875,0,0,NaN,1
2,I will murder your family friend!,Extremely violent,3,4,1,1,1,murder family friend,[],0,...,"[('murder', 'family'), ('family', 'friend')]","[('murder', 'family', 'friend')]","['murder', 'family', 'friend']",0.260667,0.164667,0.215333,0,0,NaN,1
3,"Eww, refer to me as a disgusting subhuman Celt...",Extremely violent,3,4,1,1,1,eww refer disgusting subhuman celt blood eagle...,[],0,...,"[('eww', 'refer'), ('refer', 'disgusting'), ('...","[('eww', 'refer', 'disgusting'), ('refer', 'di...","['eww', 'refer', 'disgust', 'subhuman', 'celt'...",-0.236333,0.039333,0.097333,0,0,NaN,1
4,And then he rapes you,Extremely violent,3,4,1,1,1,rapes,[],0,...,[],[],['rape'],0.000000,0.000000,0.000000,0,0,NaN,1


In [17]:
df_media_sens[["tossicita", "isToxic"]].groupby("isToxic").count()

,tossicita
isToxic,
0,134
1,933


In [ ]:
df_media_sens.head()

,commento,valutazione,tossicita,tipo_tossicita,age_group,gender,utente_id,comment_cleaned,emoji_set,emoji_count,...,comment_tok=2,comment_tok=3,comment_lemmatized,VAD_valence,VAD_arousal,VAD_dominance,bad_words_flag,bad_words_count,bad_words_matches,isToxic
0,Think carefully. Accuse me of raping kids and ...,Extremely violent,3,4,1,1,1,think carefully accuse raping kids going rape ...,[],0,...,"[('think', 'carefully'), ('carefully', 'accuse...","[('think', 'carefully', 'accuse'), ('carefully...","['think', 'carefully', 'accuse', 'rap', 'kid',...",0.074000,0.158000,0.014857,1,2,"raping, rape",1
1,Maybe someone should choke you and your immedi...,Extremely violent,3,4,1,1,1,maybe someone choke immediate family death see...,[],0,...,"[('maybe', 'someone'), ('someone', 'choke'), (...","[('maybe', 'someone', 'choke'), ('someone', 'c...","['maybe', 'someone', 'choke', 'immediate', 'fa...",0.062500,-0.008375,0.006875,0,0,NaN,1
2,I will murder your family friend!,Extremely violent,3,4,1,1,1,murder family friend,[],0,...,"[('murder', 'family'), ('family', 'friend')]","[('murder', 'family', 'friend')]","['murder', 'family', 'friend']",0.260667,0.164667,0.215333,0,0,NaN,1
3,"Eww, refer to me as a disgusting subhuman Celt...",Extremely violent,3,4,1,1,1,eww refer disgusting subhuman celt blood eagle...,[],0,...,"[('eww', 'refer'), ('refer', 'disgusting'), ('...","[('eww', 'refer', 'disgusting'), ('refer', 'di...","['eww', 'refer', 'disgust', 'subhuman', 'celt'...",-0.236333,0.039333,0.097333,0,0,NaN,1
4,And then he rapes you,Extremely violent,3,4,1,1,1,rapes,[],0,...,[],[],['rape'],0.000000,0.000000,0.000000,0,0,NaN,1


In [18]:
df_alta_sens[["tossicita", "isToxic"]].groupby("isToxic").count()

,tossicita
isToxic,
0,25
1,1042


In [19]:
df_alta_sens.head()

,commento,valutazione,tossicita,tipo_tossicita,age_group,gender,utente_id,comment_cleaned,emoji_set,emoji_count,...,comment_tok=2,comment_tok=3,comment_lemmatized,VAD_valence,VAD_arousal,VAD_dominance,bad_words_flag,bad_words_count,bad_words_matches,isToxic
0,Think carefully. Accuse me of raping kids and ...,Extremely violent,3,4,1,1,1,think carefully accuse raping kids going rape ...,[],0,...,"[('think', 'carefully'), ('carefully', 'accuse...","[('think', 'carefully', 'accuse'), ('carefully...","['think', 'carefully', 'accuse', 'rap', 'kid',...",0.074000,0.158000,0.014857,1,2,"raping, rape",1
1,Maybe someone should choke you and your immedi...,Extremely violent,3,4,1,1,1,maybe someone choke immediate family death see...,[],0,...,"[('maybe', 'someone'), ('someone', 'choke'), (...","[('maybe', 'someone', 'choke'), ('someone', 'c...","['maybe', 'someone', 'choke', 'immediate', 'fa...",0.062500,-0.008375,0.006875,0,0,NaN,1
2,I will murder your family friend!,Extremely violent,3,4,1,1,1,murder family friend,[],0,...,"[('murder', 'family'), ('family', 'friend')]","[('murder', 'family', 'friend')]","['murder', 'family', 'friend']",0.260667,0.164667,0.215333,0,0,NaN,1
3,"Eww, refer to me as a disgusting subhuman Celt...",Extremely violent,3,4,1,1,1,eww refer disgusting subhuman celt blood eagle...,[],0,...,"[('eww', 'refer'), ('refer', 'disgusting'), ('...","[('eww', 'refer', 'disgusting'), ('refer', 'di...","['eww', 'refer', 'disgust', 'subhuman', 'celt'...",-0.236333,0.039333,0.097333,0,0,NaN,1
4,And then he rapes you,Extremely violent,3,4,1,1,1,rapes,[],0,...,[],[],['rape'],0.000000,0.000000,0.000000,0,0,NaN,1
